In [ ]:
pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.2 MB/s eta 0:00:00


In [ ]:
# pip install pandas statsmodels linearmodels

import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# ------------------------------------------------------------
# 1. Load public Card dataset
# ------------------------------------------------------------
# Public mirror of the wooldridge/card dataset
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/wooldridge/card.csv"
df = pd.read_csv(url)

# Drop the unnamed index column if present
if "rownames" in df.columns:
    df = df.drop(columns=["rownames"])

# ------------------------------------------------------------
# 2. Define variables
# ------------------------------------------------------------
# Outcome: log wage
# Treatment: years of education
# Instrument: nearc4 = near 4-year college in 1966
# Controls: a standard compact set for classroom illustration
y_var = "lwage"
x_endog = "educ"
z_var = "nearc4"

controls = [
    "exper",
    "expersq",
    "black",
    "south",
    "smsa"
]

vars_needed = [y_var, x_endog, z_var] + controls
df = df[vars_needed].dropna().copy()

# Add constant for OLS / first stage / reduced form
X_controls = sm.add_constant(df[controls])

print("Sample size:", len(df))
print(df.head())


Sample size: 3010
      lwage  educ  nearc4  exper  expersq  black  south  smsa
0  6.306275     7       0     16      256      1      0     1
1  6.175867    12       0      9       81      0      0     1
2  6.580639    12       0     16      256      0      0     1
3  5.521461    11       1     10      100      0      0     1
4  6.591674    12       1     16      256      0      0     1


In [ ]:

# ------------------------------------------------------------
# 3. OLS
# ------------------------------------------------------------
X_ols = sm.add_constant(df[[x_endog] + controls])
ols = sm.OLS(df[y_var], X_ols).fit(cov_type="HC1")

print("\n" + "=" * 70)
print("OLS: lwage ~ educ + controls")
print("=" * 70)
print(ols.summary())



OLS: lwage ~ educ + controls
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.291
Model:                            OLS   Adj. R-squared:                  0.289
Method:                 Least Squares   F-statistic:                     217.7
Date:                Mon, 23 Mar 2026   Prob (F-statistic):          3.04e-231
Time:                        05:29:17   Log-Likelihood:                -1308.7
No. Observations:                3010   AIC:                             2631.
Df Residuals:                    3003   BIC:                             2673.
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.7337 

In [ ]:

# ------------------------------------------------------------
# 4. First stage
# ------------------------------------------------------------
X_fs = sm.add_constant(df[[z_var] + controls])
first_stage = sm.OLS(df[x_endog], X_fs).fit(cov_type="HC1")

print("\n" + "=" * 70)
print("First stage: educ ~ nearc4 + controls")
print("=" * 70)
print(first_stage.summary())

# Calculate predicted values of educ from the first stage
educ_hat = first_stage.predict(X_fs)

# ------------------------------------------------------------
# 5. Second stage (manual 2SLS)
# ------------------------------------------------------------
# For the second stage, we regress the outcome (lwage) on the predicted
# endogenous variable (educ_hat) and the exogenous controls.

X_second_stage = sm.add_constant(pd.DataFrame({'educ_hat': educ_hat}))
X_second_stage = X_second_stage.join(df[controls])

second_stage_manual = sm.OLS(df[y_var], X_second_stage).fit(cov_type="HC1")

print("\n" + "=" * 70)
print("Second stage (manual): lwage ~ educ_hat + controls")
print("=" * 70)
print(second_stage_manual.summary())



First stage: educ ~ nearc4 + controls
                            OLS Regression Results                            
Dep. Variable:                   educ   R-squared:                       0.474
Model:                            OLS   Adj. R-squared:                  0.473
Method:                 Least Squares   F-statistic:                     608.0
Date:                Mon, 23 Mar 2026   Prob (F-statistic):               0.00
Time:                        05:34:28   Log-Likelihood:                -6266.1
No. Observations:                3010   AIC:                         1.255e+04
Df Residuals:                    3003   BIC:                         1.259e+04
Df Model:                           6                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        

In [ ]:

# ------------------------------------------------------------
# 6. 2SLS / IV
# ------------------------------------------------------------
# Formula:
#   outcome ~ exogenous controls + [endogenous treatment ~ instrument]
formula = "lwage ~ 1 + exper + expersq + black + south + smsa + [educ ~ nearc4]"
iv = IV2SLS.from_formula(formula, data=df).fit(cov_type="robust")

print("\n" + "=" * 70)
print("2SLS / IV: instrument educ with nearc4")
print("=" * 70)
print(iv.summary)

# ------------------------------------------------------------
# 7. Wald ratio (optional, naive version without controls)
# ------------------------------------------------------------
# For teaching intuition only: Wald = reduced-form / first-stage without controls
df_naive = df[[y_var, x_endog, z_var]].dropna()
rf_naive = sm.OLS(df_naive[y_var], sm.add_constant(df_naive[[z_var]])).fit()
fs_naive = sm.OLS(df_naive[x_endog], sm.add_constant(df_naive[[z_var]])).fit()

wald = rf_naive.params[z_var] / fs_naive.params[z_var]
print("\n" + "=" * 70)
print("Naive Wald ratio without controls (for intuition only)")
print("=" * 70)
print(f"Wald estimate = {wald:.4f}")

# ------------------------------------------------------------
# 8. Short interpretation helper
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("Key quantities")
print("=" * 70)
print(f"OLS return to schooling estimate: {ols.params['educ']:.4f}")
print(f"First-stage effect of nearc4 on educ: {first_stage.params['nearc4']:.4f}")
# Note: Reduced-form is no longer explicitly calculated as a separate object, so removed here.
print(f"Manual 2SLS (second stage) return to schooling estimate: {second_stage_manual.params['educ_hat']:.4f}")
print(f"IV return to schooling estimate (from linearmodels): {iv.params['educ']:.4f}")



2SLS / IV: instrument educ with nearc4
                          IV-2SLS Estimation Summary                          
Dep. Variable:                  lwage   R-squared:                      0.2252
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2237
No. Observations:                3010   F-statistic:                    792.07
Date:                Mon, Mar 23 2026   P-value (F-stat)                0.0000
Time:                        05:34:38   Distribution:                  chi2(6)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      3.7528     0.8167     4.5948     0.0000      2.1520      5.3536
exper       